# 06b – XLM-RoBERTa Fine-Tuning

## Objective
Fine-tune XLM-RoBERTa (a multilingual transformer) for 3-class
sentiment classification on the cleaned Daraz Nepal reviews, using
the same class weights as MuRIL for a fair comparison.

## Input
- Cleaned splits from notebook 02: `train_cleaned.csv`, `val_cleaned.csv`, `test_cleaned.csv`

## Output
- Fine-tuned model saved to `xlmr_final_model_v2` on Google Drive
- Final verified test macro F1: 0.6929

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# 1. Load data (same files as MuRIL)
train_df = pd.read_csv('/content/drive/MyDrive/research_transformer/train_cleaned.csv')
val_df = pd.read_csv('/content/drive/MyDrive/research_transformer/val_cleaned.csv')
test_df = pd.read_csv('/content/drive/MyDrive/research_transformer/test_cleaned.csv')

label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
train_df['label'] = train_df['sentiment_label'].map(label_map)
val_df['label'] = val_df['sentiment_label'].map(label_map)
test_df['label'] = test_df['sentiment_label'].map(label_map)

# 2. Tokenizer (XLM-R specific, cannot reuse MuRIL's)
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_texts(texts, max_length=128):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

train_encodings = tokenize_texts(train_df['review_text'])
val_encodings = tokenize_texts(val_df['review_text'])
test_encodings = tokenize_texts(test_df['review_text'])

# 3. Dataset class (same shape as before)
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = ReviewDataset(train_encodings, train_df['label'].tolist())
val_dataset = ReviewDataset(val_encodings, val_df['label'].tolist())
test_dataset = ReviewDataset(test_encodings, test_df['label'].tolist())

# 4. Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1_macro = f1_score(labels, preds, average='macro')
    return {'f1_macro': f1_macro}

# 5. Model + class weights (same weights as MuRIL, for fair comparison)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

class_weights = torch.tensor([1.5, 1.7, 1.0]).to(model.device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = class_weights.to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir='/content/xlmr_results_v1',
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Mounted at /content/drive


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1528, training_loss=0.6353096525082413, metrics={'train_runtime': 1040.2583, 'train_samples_per_second': 23.386, 'train_steps_per_second': 1.469, 'total_flos': 1600255806621696.0, 'train_loss': 0.6353096525082413, 'epoch': 8.0})

In [ ]:
trainer.evaluate(test_dataset)

Training Loss,Validation Loss,Epoch,F1 Macro
0.443136,0.867075,8,0.688346


{'eval_loss': 0.8670746088027954, 'eval_f1_macro': 0.6883464078185302}

## Saving this time
The first run above trained successfully but was never saved before
the session ended. This retrain adds an explicit save immediately
after training completes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# 1. Load data
train_df = pd.read_csv('/content/drive/MyDrive/research_transformer/train_cleaned.csv')
val_df = pd.read_csv('/content/drive/MyDrive/research_transformer/val_cleaned.csv')
test_df = pd.read_csv('/content/drive/MyDrive/research_transformer/test_cleaned.csv')

label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
train_df['label'] = train_df['sentiment_label'].map(label_map)
val_df['label'] = val_df['sentiment_label'].map(label_map)
test_df['label'] = test_df['sentiment_label'].map(label_map)

# 2. Tokenizer
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_texts(texts, max_length=128):
    return tokenizer(list(texts), padding=True, truncation=True, max_length=max_length, return_tensors="pt")

train_encodings = tokenize_texts(train_df['review_text'])
val_encodings = tokenize_texts(val_df['review_text'])
test_encodings = tokenize_texts(test_df['review_text'])

# 3. Dataset class
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = ReviewDataset(train_encodings, train_df['label'].tolist())
val_dataset = ReviewDataset(val_encodings, val_df['label'].tolist())
test_dataset = ReviewDataset(test_encodings, test_df['label'].tolist())

# 4. Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {'f1_macro': f1_score(labels, preds, average='macro')}

# 5. Model + weighted trainer
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
class_weights = torch.tensor([1.5, 1.7, 1.0]).to(model.device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = class_weights.to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir='/content/xlmr_results_v1',
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

# SAVE IMMEDIATELY — before evaluate, before anything else
trainer.save_model('/content/drive/MyDrive/research_transformer/xlmr_final_model')
print("Model saved successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully.


In [ ]:
test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=1)
test_true = test_predictions.label_ids

print(classification_report(test_true, test_preds, target_names=['negative', 'neutral', 'positive']))
print(confusion_matrix(test_true, test_preds))

              precision    recall  f1-score   support

    negative       0.69      0.72      0.70       176
     neutral       0.59      0.57      0.58       178
    positive       0.82      0.82      0.82       298

    accuracy                           0.72       652
   macro avg       0.70      0.70      0.70       652
weighted avg       0.72      0.72      0.72       652

[[126  35  15]
 [ 38 102  38]
 [ 19  36 243]]


## Verifying the saved model
Reloading the model saved above, from a fresh session, with a
classifier weight check added first (the same untrained-head issue
found in the MuRIL notebook) before trusting any results from it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import numpy as np
import pandas as pd

MODEL_PATH = '/content/drive/MyDrive/research_transformer/xlmr_final_model'

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

# Diagnostic check FIRST, before trusting anything
classifier_std = model.classifier.out_proj.weight.data.std().item() if hasattr(model.classifier, 'out_proj') else model.classifier.weight.data.std().item()
print(f"Classifier weight std: {classifier_std:.4f} (if ~0.02, this is also untrained)")

test_df = pd.read_csv('/content/drive/MyDrive/research_transformer/test_cleaned.csv')
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
test_df['label'] = test_df['sentiment_label'].map(label_map)

test_encodings = tokenizer(list(test_df['review_text']), truncation=True, padding='max_length', max_length=128, return_tensors='pt')

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

test_dataset = ReviewDataset(test_encodings, list(test_df['label']))
trainer = Trainer(model=model)

test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=1)
test_true = test_predictions.label_ids

print("\n=== XLM-R TEST SET RESULTS ===")
print(classification_report(test_true, test_preds, target_names=['negative', 'neutral', 'positive'], digits=4))
print("Macro F1:", f1_score(test_true, test_preds, average='macro'))
print(confusion_matrix(test_true, test_preds))

Mounted at /content/drive


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Classifier weight std: 0.0199 (if ~0.02, this is also untrained)



=== XLM-R TEST SET RESULTS ===
              precision    recall  f1-score   support

    negative     0.2619    0.3750    0.3084       176
     neutral     0.2959    0.6517    0.4070       178
    positive     0.3750    0.0101    0.0196       298

    accuracy                         0.2837       652
   macro avg     0.3109    0.3456    0.2450       652
weighted avg     0.3229    0.2837    0.2033       652

Macro F1: 0.24501220065005835
[[ 66 108   2]
 [ 59 116   3]
 [127 168   3]]


The reload above confirmed the same untrained-head bug seen with
MuRIL (classifier weight std ~0.02, the default random-initialization
value) despite training completing normally. This cell retrains from
scratch and includes the classifier weight check immediately after
training, before saving, so the save can be trusted this time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# 1. Load data
train_df = pd.read_csv('/content/drive/MyDrive/research_transformer/train_cleaned.csv')
val_df = pd.read_csv('/content/drive/MyDrive/research_transformer/val_cleaned.csv')
test_df = pd.read_csv('/content/drive/MyDrive/research_transformer/test_cleaned.csv')

label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
train_df['label'] = train_df['sentiment_label'].map(label_map)
val_df['label'] = val_df['sentiment_label'].map(label_map)
test_df['label'] = test_df['sentiment_label'].map(label_map)

# 2. Tokenize (XLM-R tokenizer, not MuRIL's)
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
train_encodings = tokenizer(list(train_df['review_text']), truncation=True, padding='max_length', max_length=128)
val_encodings = tokenizer(list(val_df['review_text']), truncation=True, padding='max_length', max_length=128)
test_encodings = tokenizer(list(test_df['review_text']), truncation=True, padding='max_length', max_length=128)

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = ReviewDataset(train_encodings, list(train_df['label']))
val_dataset = ReviewDataset(val_encodings, list(val_df['label']))
test_dataset = ReviewDataset(test_encodings, list(test_df['label']))

# 3. Fresh model
model = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=3)

# 4. Class weights
class_weights = torch.tensor([1.5, 1.7, 1.0]).to(model.device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = class_weights.to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"f1_macro": f1_score(labels, preds, average='macro')}

training_args = TrainingArguments(
    output_dir='/content/xlmr_results_v2',
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# 5. Train
trainer.train()

# ============================================================
# EVERYTHING BELOW RUNS IMMEDIATELY, SAME CELL, NO GAP
# ============================================================

classifier_std = trainer.model.classifier.out_proj.weight.data.std().item()
print(f"\nClassifier weight std after training: {classifier_std:.4f} (should be well above 0.02)\n")

test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=1)
test_true = test_predictions.label_ids

print("=== XLM-R TEST SET RESULTS ===")
print(classification_report(test_true, test_preds, target_names=['negative', 'neutral', 'positive'], digits=4))
print("Macro F1:", f1_score(test_true, test_preds, average='macro'))
print(confusion_matrix(test_true, test_preds))

SAVE_PATH = '/content/drive/MyDrive/research_transformer/xlmr_final_model_v2'
trainer.save_model(SAVE_PATH)
print(f"\nSaved to {SAVE_PATH}")

reload_check = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH)
reload_std = reload_check.classifier.out_proj.weight.data.std().item()
print(f"Reloaded classifier std: {reload_std:.4f} (should match this cell's training std, NOT ~0.02)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

=== XLM-R TEST SET RESULTS ===
              precision    recall  f1-score   support

    negative     0.6872    0.7614    0.7224       176
     neutral     0.5833    0.5112    0.5449       178
    positive     0.8073    0.8154    0.8114       298

    accuracy                         0.7178       652
   macro avg     0.6926    0.6960    0.6929       652
weighted avg     0.7137    0.7178    0.7146       652

Macro F1: 0.6928781336839885
[[134  28  14]
 [ 43  91  44]
 [ 18  37 243]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved to /content/drive/MyDrive/research_transformer/xlmr_final_model_v2


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Reloaded classifier std: 0.0203 (should match this cell's training std, NOT ~0.02)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import f1_score
import csv

MODEL_PATH = '/content/drive/MyDrive/research_transformer/xlmr_final_model_v2'

# Load tokenizer from the ORIGINAL source, not from the saved folder
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

test_df = pd.read_csv('/content/drive/MyDrive/research_transformer/test_cleaned.csv')
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
test_df['label'] = test_df['sentiment_label'].map(label_map)

test_encodings = tokenizer(list(test_df['review_text']), truncation=True, padding=True, max_length=128, return_tensors="pt")

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

test_dataset = ReviewDataset(test_encodings, list(test_df['label']))
trainer = Trainer(model=model)

test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=1)
test_true = test_predictions.label_ids

macro_f1_value = f1_score(test_true, test_preds, average='macro')
print("XLM-RoBERTa Macro F1:", macro_f1_value)

# APPEND this time, not overwrite, so MuRIL's row stays
with open('/content/drive/MyDrive/research_transformer/transformer_results.csv', 'a', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["XLM-RoBERTa", macro_f1_value])

print("Saved to transformer_results.csv")

Mounted at /content/drive


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

XLM-RoBERTa Macro F1: 0.6928781336839885
Saved to transformer_results.csv


In [ ]:
import csv

xlmr_class_f1 = f1_score(test_true, test_preds, average=None, labels=[0, 1, 2])
print("XLM-RoBERTa per-class F1 (negative, neutral, positive):", xlmr_class_f1)

with open('/content/drive/MyDrive/research_transformer/transformer_per_class_f1.csv', 'a', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['XLM-RoBERTa', xlmr_class_f1[0], xlmr_class_f1[1], xlmr_class_f1[2]])

print("Saved.")

XLM-RoBERTa per-class F1 (negative, neutral, positive): [0.72237197 0.54491018 0.81135225]
Saved.


In [ ]:
import pandas as pd
check_df = pd.read_csv('/content/drive/MyDrive/research_transformer/transformer_per_class_f1.csv')
print(check_df)

Empty DataFrame
Columns: [XLM-RoBERTa, 0.7223719676549866, 0.5449101796407185, 0.8113522537562604]
Index: []


In [ ]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import f1_score
import csv

# Step 1: Start the file fresh with a proper header
with open('/content/drive/MyDrive/research_transformer/transformer_per_class_f1.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['model', 'negative_f1', 'neutral_f1', 'positive_f1'])

# Step 2: Run MuRIL and append its row
MODEL_PATH = '/content/drive/MyDrive/research_transformer/muril_final_model_v2'
tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

test_df = pd.read_csv('/content/drive/MyDrive/research_transformer/test_cleaned.csv')
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
test_df['label'] = test_df['sentiment_label'].map(label_map)

test_encodings = tokenizer(list(test_df['review_text']), truncation=True, padding=True, max_length=128, return_tensors="pt")

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

test_dataset = ReviewDataset(test_encodings, list(test_df['label']))
trainer = Trainer(model=model)
test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=1)
test_true = test_predictions.label_ids

muril_class_f1 = f1_score(test_true, test_preds, average=None, labels=[0, 1, 2])
print("MuRIL:", muril_class_f1)

with open('/content/drive/MyDrive/research_transformer/transformer_per_class_f1.csv', 'a', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['MuRIL', muril_class_f1[0], muril_class_f1[1], muril_class_f1[2]])

# Step 3: Run XLM-R and append its row
MODEL_PATH = '/content/drive/MyDrive/research_transformer/xlmr_final_model_v2'
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

test_encodings = tokenizer(list(test_df['review_text']), truncation=True, padding=True, max_length=128, return_tensors="pt")
test_dataset = ReviewDataset(test_encodings, list(test_df['label']))
trainer = Trainer(model=model)
test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=1)
test_true = test_predictions.label_ids

xlmr_class_f1 = f1_score(test_true, test_preds, average=None, labels=[0, 1, 2])
print("XLM-RoBERTa:", xlmr_class_f1)

with open('/content/drive/MyDrive/research_transformer/transformer_per_class_f1.csv', 'a', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['XLM-RoBERTa', xlmr_class_f1[0], xlmr_class_f1[1], xlmr_class_f1[2]])

# Step 4: Verify
final_check = pd.read_csv('/content/drive/MyDrive/research_transformer/transformer_per_class_f1.csv')
print(final_check)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

MuRIL: [0.68208092 0.52873563 0.82295082]


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLM-RoBERTa: [0.72237197 0.54491018 0.81135225]
         model  negative_f1  neutral_f1  positive_f1
0        MuRIL     0.682081    0.528736     0.822951
1  XLM-RoBERTa     0.722372    0.544910     0.811352
